# Yeast promoter design with TACO on Google Colab, using PyTorch

This notebook prepares the published TACO code<sup>1</sup> so that it can run on Google Colab. **Part 1** holds the setup steps only: imports, settings, the graphics card, Google Drive, unpacking the code, checking the files, and finding the file paths that must be changed.

Not written yet: downloading the model and data files from Hugging Face, changing the file paths, the tiny test run with its timing, and the extra checker networks. These parts are added after the Hugging Face file lists are known.

### **Step 1: Package Import**

In [19]:
from importlib.metadata import version, PackageNotFoundError
import pathlib
import platform
import shutil
import subprocess
import zipfile
import requests
import tarfile
from huggingface_hub import HfApi


print("Python==" + platform.python_version())

packages = ["torch", "numpy", "pandas", "pathlib", "platform", "shutil", "subprocess", "zipfile", "requests", "huggingface_hub"]

for package in packages:
    try:
        print(package + "==" + version(package))
    except PackageNotFoundError:
        print(package + ": not installed yet")

Python==3.13.16
torch==2.14.1+cu130
numpy==2.4.6
pandas==2.2.3
pathlib: not installed yet
platform: not installed yet
shutil: not installed yet
subprocess: not installed yet
zipfile: not installed yet
requests==2.32.4
huggingface_hub==1.33.0


### **Step 2: Settings**

All names and folders are written down in this one cell. If a name changes, it is changed here only.

In [2]:
# This is the only cell that needs future editing.

# The dir in my Google Drive that holds all big files.
drive_dir = "Reinforcement_Learning_for_Promoter_Design"

# The subfolder of that Drive dir, and the name of the zip file inside it.
code_subfolder = "taco_code"
taco_zip_file = "TACO-main.zip"

# The working folder for this session. Colab erases this folder when the session ends.
working_folder = "/content/working"

# Only for testing outside Colab, for folder that stands in for the drive dir, locally:
local_dir = ""

### **Step 3: Check the graphics card**

TACO is written for a graphics card from NVIDIA, so it cannot run without one. The command `nvidia-smi` asks the computer which card it has.

In [3]:
# HELPER FUNCTION
# Asking the computer for its graphics card and printing the answer.
def show_graphics_card_information():
    try:
        completed_process = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
        print(completed_process.stdout)
    except FileNotFoundError:
        print("No graphics card found. On Colab: Runtime, Change runtime type, choose a graphics card.")

show_graphics_card_information()

Fri Nov 13 11:26:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### **Step 4: Connect Google Drive**

Colab erases its own files when the session ends, but Drive keeps them. So the zip, the downloaded model files and the results all live in the Drive folder.

In [6]:
# HELPER FUNCTION
# Connecting Drive on Colab, and returning the project folder in it.
def find_drive_dir():
    if local_dir != "":
        # Only used for testing outside Colab.
        return pathlib.Path(local_dir)
    from google.colab import drive
    drive.mount("/content/drive")
    return pathlib.Path("/content/drive/MyDrive") / drive_dir

drive_dir_name = find_drive_dir()
print("Drive project folder:", drive_dir_name)
print("Folder exists:", drive_dir_name.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive project folder: /content/drive/MyDrive/Reinforcement_Learning_for_Promoter_Design
Folder exists: True


### **Step 5: Unpack the TACO zip into the working folder**

The zip is copied from Drive and unpacked in the working folder. Working from the session's own disk is faster than working from Drive, and the zip in Drive stays untouched. If the working folder already exists, it is deleted first, so that old changes are never mixed in.

In [7]:
taco_zip_path = drive_dir_name / code_subfolder / taco_zip_file
working_folder_path = pathlib.Path(working_folder)

# Stopping early, with a clear message, if the zip is not where Step 2 says.
if not taco_zip_path.exists():
    raise FileNotFoundError("The zip was not found at " + str(taco_zip_path) + ". Check the names in Step 2.")

if working_folder_path.exists():
    shutil.rmtree(working_folder_path)
working_folder_path.mkdir(parents=True)

with zipfile.ZipFile(taco_zip_path) as zip_file:
    zip_file.extractall(working_folder_path)
    print("Files unpacked:", len(zip_file.namelist()))

taco_folder = working_folder_path / "TACO-main"
print("Unpacked folder:", taco_folder, "| exists:", taco_folder.exists())

Files unpacked: 117
Unpacked folder: /content/working/TACO-main | exists: True


### **Step 6: Check that the expected files are there**

This looks for the files that the TACO instructions and code mention, to find out early whether the zip is complete.

In [8]:
expected_files = [
    "README.md",
    "reinforce_mbo.py",
    "reinforce_mbo.sh",
    "env_install.sh",
    "dna_optimizers/mbo_optimizaer.py",
    "dna_optimizers/experience.py",
    "reglm_src/reglm/regression.py",
    "reglm_src/reglm/lightning.py",
]
for relative_file_name in expected_files:
    print(relative_file_name, (taco_folder / relative_file_name).exists())

README.md True
reinforce_mbo.py True
reinforce_mbo.sh True
env_install.sh True
dna_optimizers/mbo_optimizaer.py True
dna_optimizers/experience.py True
reglm_src/reglm/regression.py True
reglm_src/reglm/lightning.py True


### **Step 7: Find the file paths that belong to the authors' computer**

The TACO code reads its models and data from `/blob/...`, a folder on the authors' own computer. Each of these paths must later point to a real file in your Drive folder. This step lists them, so that we know how much has to be changed. Lines that are comments are skipped.

In [9]:
# HELPER FUNCTION
# Going through every Python file in the folder and collecting the lines that contain the given text.
def find_lines_with_text(folder, search_text):
    found_lines = []
    for python_file in sorted(folder.rglob("*.py")):
        file_text = python_file.read_text(encoding="utf-8", errors="replace")
        for line_number, line_text in enumerate(file_text.splitlines(), start=1):
            if search_text in line_text and not line_text.strip().startswith("#"):
                found_lines.append((python_file.relative_to(folder), line_number, line_text.strip()))
    return found_lines

blob_lines = find_lines_with_text(taco_folder, "/blob/")
for file_name, line_number, line_text in blob_lines:
    print(file_name, "line", line_number, ":", line_text)
print("Total lines with /blob/:", len(blob_lines))

aggregate_mbo.py line 51 : inits = pd.read_csv(f'/blob/Data/DNA/Design/data/{cell}/{level}.csv')
dna_optimizers/mbo_optimizaer.py line 166 : "/blob/Data/DNA/Design/guide_model/yeast_regression_paired_complex_offline.ckpt",
dna_optimizers/mbo_optimizaer.py line 171 : "/blob/Data/DNA/Design/guide_model/yeast_regression_paired_defined_offline.ckpt",
dna_optimizers/mbo_optimizaer.py line 177 : "/blob/Data/DNA/Design/guide_model/human_regression_paired_hepg2_mbo.ckpt",
dna_optimizers/mbo_optimizaer.py line 183 : "/blob/Data/DNA/Design/guide_model/human_regression_paired_k562_mbo.ckpt",
dna_optimizers/mbo_optimizaer.py line 190 : "/blob/Data/DNA/Design/guide_model/human_regression_paired_sknsh_mbo.ckpt",
dna_optimizers/mbo_optimizaer.py line 201 : "/blob/ICLR/oracle/yeast_regression_paired_complex.ckpt",
dna_optimizers/mbo_optimizaer.py line 206 : "/blob/ICLR/oracle/yeast_regression_paired_defined.ckpt",
dna_optimizers/mbo_optimizaer.py line 212 : "/blob/ICLR/oracle/human_regression_paired_h

### **Step 8: List the files on the Hugging Face pages of TACO**

The TACO authors say that their prepared data and model files are on Hugging Face. We need to list the file names and sizes, to download them later.The output list will tells which files exist, how large they are, and which files the code needs that are missing.

In [12]:
# Asking Hugging Face for the files on the two TACO pages: the model page and the dataset page.
hugging_face_interface = HfApi()

for repository_type in ["model", "dataset"]:
    if repository_type == "model":
        repository_information = hugging_face_interface.model_info("yangyz1230/TACO", files_metadata=True)
    else:
        repository_information = hugging_face_interface.dataset_info("yangyz1230/TACO", files_metadata=True)

    print("==", repository_type, "yangyz1230/TACO ==")
    for file_information in repository_information.siblings:
        size_in_megabytes = None if file_information.size is None else round(file_information.size / 1e6, 1)
        print(file_information.rfilename, "|", size_in_megabytes, "megabytes")

== model yangyz1230/TACO ==
.gitattributes | 0.0 megabytes
README.md | 0.0 megabytes
human_regression_paired_hepg2_mbo.ckpt | 2758.6 megabytes
human_regression_paired_k562_mbo.ckpt | 2758.6 megabytes
human_regression_paired_sknsh_mbo.ckpt | 2758.6 megabytes
== dataset yangyz1230/TACO ==
.gitattributes | 0.0 megabytes
README.md | 0.0 megabytes
hepg2/mbo.csv | 92.1 megabytes
k562/mbo.csv | 92.1 megabytes
reward_dict/results_hepg2_mbo.json | 0.0 megabytes
reward_dict/results_k562_mbo.json | 0.0 megabytes
reward_dict/results_sknsh_mbo.json | 0.0 megabytes
sknsh/mbo.csv | 92.1 megabytes


### **Step 9: List the files of the regLM record on Zenodo**

The TACO README says that the writer and the oracle models come from regLM. This step only **lists** the files of that record, with their sizes.

In [17]:
# Asking Zenodo for the description of the regLM record that the TACO README mentions.
zenodo_response = requests.get("https://zenodo.org/api/records/12668907", timeout=60)
zenodo_record = zenodo_response.json()

print("Title:", zenodo_record["metadata"]["title"])
for file_information in zenodo_record["files"]:
    size_in_megabytes = round(file_information["size"] / 1e6, 1)
    print(file_information["key"], "|", size_in_megabytes, "megabytes")

Title: Designing realistic regulatory DNA with autoregressive language models
scripts.tar.gz | 0.0 megabytes
yeast_promoters.tar.gz | 270.5 megabytes
other_human_models.tar.gz | 0.0 megabytes
human_enhancers.tar.gz | 15361.0 megabytes


### **Step 10: List the contents of the yeast archive of regLM**

The yeast archive on Zenodo is 270.5 megabytes. This step reads it as a stream, without saving it, and prints the names and sizes of the files inside. The list shows whether the yeast writer and the yeast oracle are in it.

In [20]:
# Reading the archive straight from Zenodo, piece by piece, and printing what is inside.
yeast_archive_address = "https://zenodo.org/records/12668907/files/yeast_promoters.tar.gz?download=1"

with requests.get(yeast_archive_address, stream=True, timeout=120) as zenodo_download:
    zenodo_download.raise_for_status()
    with tarfile.open(fileobj=zenodo_download.raw, mode="r|gz") as archive:
        for member in archive:
            print(member.name, "|", round(member.size / 1e6, 1), "megabytes")

yeast_promoters | 0.0 megabytes
yeast_promoters/09_synthetic_promoter_comparison | 0.0 megabytes
yeast_promoters/09_synthetic_promoter_comparison/05_comparison_embedding.ipynb | 0.0 megabytes
yeast_promoters/09_synthetic_promoter_comparison/02_comparison_sequence.ipynb | 0.0 megabytes
yeast_promoters/09_synthetic_promoter_comparison/03_comparison_motifs.ipynb | 0.0 megabytes
yeast_promoters/09_synthetic_promoter_comparison/01_match_generated.ipynb | 0.0 megabytes
yeast_promoters/09_synthetic_promoter_comparison/04_comparison_pair_rules.ipynb | 0.0 megabytes
yeast_promoters/01_data_processing | 0.0 megabytes
yeast_promoters/01_data_processing/06_categorize-10.ipynb | 0.0 megabytes
yeast_promoters/01_data_processing/04_scan_native.ipynb | 0.0 megabytes
yeast_promoters/01_data_processing/02_categorize.ipynb | 0.0 megabytes
yeast_promoters/01_data_processing/01_process.ipynb | 0.0 megabytes
yeast_promoters/01_data_processing/05_scan_test.ipynb | 0.0 megabytes
yeast_promoters/01_data_proces

### **Step 11: Download and unpack the yeast archive of regLM**

The yeast archive is saved once in the `model_files` folder in Drive, and then unpacked there. If it is already saved, it is not downloaded again. At the end, the step lists every model file (`.ckpt`) that was unpacked.

In [22]:
# Saving the archive in Drive, unless it is already there.
yeast_archive_path = drive_dir_name / "model_files" / "yeast_promoters.tar.gz"

# Ensure the parent directories exist
yeast_archive_path.parent.mkdir(parents=True, exist_ok=True)

if not yeast_archive_path.exists():
    with requests.get(yeast_archive_address, stream=True, timeout=120) as zenodo_download:
        zenodo_download.raise_for_status()
        with open(yeast_archive_path, "wb") as archive_file:
            shutil.copyfileobj(zenodo_download.raw, archive_file)

print("Archive size in megabytes:", round(yeast_archive_path.stat().st_size / 1e6, 1))

# Unpacking it next to the archive. filter="data" refuses unsafe file paths inside the archive.
with tarfile.open(yeast_archive_path) as archive:
    archive.extractall(drive_dir_name / "model_files", filter="data")

# Listing the model files.
for checkpoint_file in sorted((drive_dir_name / "model_files").rglob("*.ckpt")):
    print(checkpoint_file.relative_to(drive_dir_name), "|", round(checkpoint_file.stat().st_size / 1e6, 1), "megabytes")

Archive size in megabytes: 270.5
model_files/yeast_promoters/02_regression_paired/yeast_regression_paired_complex.ckpt | 34.4 megabytes
model_files/yeast_promoters/02_regression_paired/yeast_regression_paired_defined.ckpt | 34.4 megabytes
model_files/yeast_promoters/03_regression_separate/yeast_regression_separate_complex.ckpt | 34.4 megabytes
model_files/yeast_promoters/03_regression_separate/yeast_regression_separate_defined.ckpt | 34.4 megabytes
model_files/yeast_promoters/04_reglm/yeast_reglm.ckpt | 78.9 megabytes
model_files/yeast_promoters/04_reglm/yeast_reglm_10.ckpt | 78.9 megabytes
